# 📊 Dashboard Qualité de Service — SNCB / Infrabel

**Auteur** : Tahar Guenfoud  
**Source** : Open Data Infrabel  
**Stack** : Python · Pandas · Plotly · ydata-profiling

> ⚠️ Ce fichier est le notebook de **RÉFÉRENCE COMPLET**. Il contient toutes les étapes du projet.

---
## 0. Imports & Configuration

In [1]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import os

# Renderer Plotly — 'iframe' fonctionne dans tous les environnements Jupyter
pio.renderers.default = 'iframe'

RAW_DIR   = 'data/raw'
CLEAN_DIR = 'data/clean'
os.makedirs(CLEAN_DIR, exist_ok=True)

print('✅ Imports OK')

✅ Imports OK


---
# ÉTAPE 1 — Extract
Lecture des 5 datasets depuis le dossier `data/raw/`.

In [2]:
DATASETS = {
    'ponctualite_par_gare'    : 'ponctualite_par_gare.csv',
    'causes_retards'          : 'causes_retards.csv',
    'ponctualite_par_moment'  : 'ponctualite_par_moment.csv',
    'trains_supprimes'        : 'trains_supprimes.csv',
    'kpi_contrat_performance' : 'kpi_contrat_performance.csv',
}

dfs = {}
for name, filename in DATASETS.items():
    path = os.path.join(RAW_DIR, filename)
    dfs[name] = pd.read_csv(path, sep=';')
    print(f'✅ {name} — {dfs[name].shape[0]} lignes × {dfs[name].shape[1]} colonnes')

✅ ponctualite_par_gare — 27343 lignes × 13 colonnes
✅ causes_retards — 425 lignes × 14 colonnes
✅ ponctualite_par_moment — 484 lignes × 9 colonnes
✅ trains_supprimes — 73 lignes × 7 colonnes
✅ kpi_contrat_performance — 177 lignes × 13 colonnes


---
# ÉTAPE 2 — Transform
Nettoyage et renommage des colonnes pour chaque dataset.

### 2.1 — Ponctualité par Gare (13 colonnes)

In [3]:
df_gare = dfs['ponctualite_par_gare'].copy()

# Renommage des 13 colonnes dans l'ordre exact du CSV
df_gare.columns = [
    'date',
    'nom_gare_fr', 'nom_gare_nl', 'nom_gare_de',
    'id_point_operationnel',
    'classification_nl', 'classification_fr', 'classification_en',
    'ponctualite_pct',
    'nb_trains', 'nb_trains_ponctuels',
    'geo_point', 'geo_shape'
]

# Parser les dates
df_gare['date'] = pd.to_datetime(df_gare['date'], format='%Y-%m')

# Supprimer les colonnes inutiles
df_gare = df_gare.drop(columns=[
    'nom_gare_nl', 'nom_gare_de',
    'id_point_operationnel',
    'classification_nl', 'classification_fr', 'classification_en',
    'geo_point', 'geo_shape'
])

# Colonne calculée : trains en retard
df_gare['nb_trains_retard'] = df_gare['nb_trains'] - df_gare['nb_trains_ponctuels']

print(f'✅ df_gare — {df_gare.shape[0]} lignes × {df_gare.shape[1]} colonnes')
display(df_gare.head())
df_gare.info()

✅ df_gare — 27343 lignes × 6 colonnes


,date,nom_gare_fr,ponctualite_pct,nb_trains,nb_trains_ponctuels,nb_trains_retard
0,2023-11-01,BEIGNEE,91.200000,250.0,228.0,22.0
1,2023-11-01,BELSELE,83.060453,1588.0,1319.0,269.0
2,2023-11-01,BERLAAR,88.265746,1159.0,1023.0,136.0
3,2023-11-01,ANTWERPEN-LUCHTBAL,80.809077,2027.0,1638.0,389.0
4,2023-11-01,AARSELE,74.226804,97.0,72.0,25.0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 27343 entries, 0 to 27342
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype         
---  ------               --------------  -----         
 0   date                 27343 non-null  datetime64[ns]
 1   nom_gare_fr          27343 non-null  object        
 2   ponctualite_pct      27343 non-null  float64       
 3   nb_trains            27343 non-null  float64       
 4   nb_trains_ponctuels  27343 non-null  float64       
 5   nb_trains_retard     27343 non-null  float64       
dtypes: datetime64[ns](1), float64(4), object(1)
memory usage: 1.3+ MB


### 2.2 — Causes des Retards (14 colonnes)

In [4]:
df_causes = dfs['causes_retards'].copy()

# Renommage des 14 colonnes
df_causes.columns = [
    'annee', 'date', 'mois',
    'responsable_nl', 'responsable', 'responsable_en',
    'nb_retards', 'nb_trains_total',
    'perte_ponctualite', 'pct_proportion',
    'nb_retards_ytd', 'nb_trains_ytd',
    'perte_ponctualite_ytd', 'pct_proportion_ytd'
]

df_causes['date'] = pd.to_datetime(df_causes['date'], format='%Y-%m')

# Garder uniquement les colonnes utiles
df_causes = df_causes[['date', 'responsable', 'nb_retards', 'nb_trains_total', 'perte_ponctualite', 'pct_proportion']]

print(f'✅ df_causes — {df_causes.shape[0]} lignes × {df_causes.shape[1]} colonnes')
display(df_causes.head())

✅ df_causes — 425 lignes × 6 colonnes


,date,responsable,nb_retards,nb_trains_total,perte_ponctualite,pct_proportion
0,2026-01-01,Robustesse systémique,1440.909241,107866,1.34,17.56
1,2026-01-01,Tiers,1748.080249,107866,1.62,21.31
2,2026-01-01,Autres,271.613784,107866,0.25,3.31
3,2026-01-01,Infrabel,945.015104,107866,0.88,11.52
4,2026-01-01,SNCB,3798.381622,107866,3.52,46.30


### 2.3 — Ponctualité par Moment (9 colonnes)

In [5]:
df_moment = dfs['ponctualite_par_moment'].copy()

# Renommage des 9 colonnes
df_moment.columns = [
    'date', 'periode_nl', 'periode', 'periode_en',
    'ponctualite_pct',
    'nb_trains', 'nb_trains_ponctuels',
    'nb_minutes_retard', 'annee'
]

df_moment['date'] = pd.to_datetime(df_moment['date'], format='%Y-%m')

# Supprimer les colonnes redondantes
df_moment = df_moment.drop(columns=['periode_nl', 'periode_en', 'annee'])

# Colonne calculée
df_moment['nb_trains_retard'] = df_moment['nb_trains'] - df_moment['nb_trains_ponctuels']

print(f'✅ df_moment — {df_moment.shape[0]} lignes × {df_moment.shape[1]} colonnes')
print('Périodes :', df_moment['periode'].unique())
display(df_moment.head())

✅ df_moment — 484 lignes × 7 colonnes
Périodes : ['Weekends' 'Heures creuses' 'Pointe du soir' 'Pointe du matin']


,date,periode,ponctualite_pct,nb_trains,nb_trains_ponctuels,nb_minutes_retard,nb_trains_retard
0,2016-01-01,Weekends,93.869404,24549,23044,32326,1505
1,2016-01-01,Heures creuses,90.689997,45435,41205,93564,4230
2,2016-02-01,Pointe du soir,88.282383,17239,15219,40597,2020
3,2016-03-01,Pointe du matin,88.131496,16548,14584,40055,1964
4,2016-03-01,Pointe du soir,88.470725,17165,15186,41783,1979


### 2.4 — Trains Supprimés (7 colonnes)

In [6]:
df_suppression = dfs['trains_supprimes'].copy()

# Renommage des 7 colonnes
df_suppression.columns = [
    'date',
    'nb_trains_supprimes_total',
    'nb_trains_supprimes_partiel',
    'nb_trains_supprimes_total_2',
    'nb_trains_planifies',
    'pct_trains_supprimes',
    'annee'
]

df_suppression['date'] = pd.to_datetime(df_suppression['date'], format='%Y-%m')
df_suppression = df_suppression.drop(columns=['annee', 'nb_trains_supprimes_total_2'])

# Fiabilité = 100% - % trains supprimés
df_suppression['fiabilite_pct'] = 100 - df_suppression['pct_trains_supprimes']

print(f'✅ df_suppression — {df_suppression.shape[0]} lignes × {df_suppression.shape[1]} colonnes')
display(df_suppression.head())

✅ df_suppression — 73 lignes × 6 colonnes


,date,nb_trains_supprimes_total,nb_trains_supprimes_partiel,nb_trains_planifies,pct_trains_supprimes,fiabilite_pct
0,2020-10-01,1920,1405,97959,1.922326,98.077674
1,2020-11-01,1920,1507,88245,2.129429,97.870571
2,2021-01-01,2245,1645,96572,2.271876,97.728124
3,2021-03-01,2722,2055,96579,2.741161,97.258839
4,2021-05-01,2282,1689,94959,2.346747,97.653253


### 2.5 — KPIs Contrat de Performance (13 colonnes)

In [7]:
df_kpi = dfs['kpi_contrat_performance'].copy()

# Renommage des 13 colonnes
df_kpi.columns = [
    'annee', 'id_indicateur', 'type',
    'categorie_nl', 'categorie',
    'sous_categorie_en', 'sous_categorie',
    'objectif', 'valeur', 'bonus',
    'remediation', 'seuil_superieur', 'unite'
]

# Garder uniquement les colonnes utiles
df_kpi = df_kpi[['annee', 'categorie', 'sous_categorie', 'objectif', 'valeur', 'unite']].dropna(subset=['valeur'])

print(f'✅ df_kpi — {df_kpi.shape[0]} lignes × {df_kpi.shape[1]} colonnes')
display(df_kpi.head())

✅ df_kpi — 132 lignes × 6 colonnes


,annee,categorie,sous_categorie,objectif,valeur,unite
45,2024,Chiffres-clés,Evolution des train-tonne-km (passagers),NaN,2.960408e+10,tonkm
46,2024,Chiffres-clés,Indice de productivité interne,NaN,1.056000e+01,€ (2024)/trkm
47,2024,Safety,Sécurité du personnel d'Infrabel,0.0,1.000000e-01,FWI
48,2024,Safety,Taux d'équipement du réseau en ETCS,80.0,7.900000e+01,%
49,2024,Safety,Risque sociétal global,NaN,1.420000e+01,FWI


---
# ÉTAPE 3 — EDA (Exploratory Data Analysis)

### 3.1 — Tendance Temporelle
> La ponctualité nationale s'améliore-t-elle au fil des années ?

In [8]:
df_tendance = df_gare.groupby('date')['ponctualite_pct'].mean().reset_index()

fig = px.line(
    df_tendance,
    x='date', y='ponctualite_pct',
    title='📈 Tendance de la Ponctualité Nationale',
    labels={'date': 'Mois', 'ponctualite_pct': 'Ponctualité (%)'},
)
fig.add_hline(y=90, line_dash='dash', line_color='red', annotation_text='Objectif 90%')
fig.show()

### 3.2 — Top 10 Gares Problématiques
> Quelles gares cumulent le plus de retards ?

In [9]:
df_analyse_gares = df_gare.groupby('nom_gare_fr').agg(
    nb_trains_retard=('nb_trains_retard', 'sum'),
    ponctualite_pct=('ponctualite_pct', 'mean')
).reset_index()

top_10 = df_analyse_gares.sort_values('nb_trains_retard', ascending=False).head(10)
top_10['ponctualite_pct'] = top_10['ponctualite_pct'].round(2)

fig = px.bar(
    top_10,
    x='nb_trains_retard', y='nom_gare_fr',
    orientation='h',
    color='ponctualite_pct',
    color_continuous_scale='RdYlGn',
    title='🚆 Top 10 Gares — Volume de Retards vs Ponctualité',
    labels={'nb_trains_retard': 'Trains en retard (total)', 'nom_gare_fr': 'Gare', 'ponctualite_pct': 'Ponctualité %'},
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

print('\n🏆 Top 3 gares les plus problématiques :')
display(top_10.head(3))


🏆 Top 3 gares les plus problématiques :


,nom_gare_fr,nb_trains_retard,ponctualite_pct
103,BRUSSEL-ZUID,170414.0,86.74
95,BRUSSEL-CENTRAAL,161951.0,87.98
100,BRUSSEL-NOORD,160342.0,88.19


### 3.3 — Analyse par Moment
> Matin vs Soir vs Heures creuses vs Weekend

In [10]:
df_par_periode = df_moment.groupby('periode').agg(
    ponctualite_pct=('ponctualite_pct', 'mean'),
    nb_trains_retard=('nb_trains_retard', 'sum')
).reset_index().round(2).sort_values('ponctualite_pct')

display(df_par_periode)

fig = px.bar(
    df_par_periode,
    x='periode', y='ponctualite_pct',
    color='ponctualite_pct',
    color_continuous_scale='RdYlGn',
    title='⏰ Ponctualité par Période',
    labels={'ponctualite_pct': 'Ponctualité (%)', 'periode': 'Période'},
    text='ponctualite_pct'
)
fig.add_hline(y=90, line_dash='dash', line_color='red', annotation_text='Objectif 90%')
fig.show()

pire = df_par_periode.iloc[0]
print(f'\n🚨 Période la plus critique : {pire["periode"]} ({pire["ponctualite_pct"]}%)')

,periode,ponctualite_pct,nb_trains_retard
2,Pointe du soir,85.99,283010
1,Pointe du matin,89.50,202093
0,Heures creuses,89.92,647525
3,Weekends,93.37,181648



🚨 Période la plus critique : Pointe du soir (85.99%)


### 3.4 — Rapport EDA Automatique (ydata-profiling)
> Scan complet du dataset gares — prend 1-2 minutes.

In [11]:
from ydata_profiling import ProfileReport

profile = ProfileReport(df_gare, title='Rapport EDA — Gares Infrabel', explorative=True)
profile.to_file('data/rapport_eda_gares.html')
print('✅ Rapport sauvegardé → data/rapport_eda_gares.html')
print('👉 Ouvre ce fichier dans ton navigateur pour explorer.')

/home/tahar/Code/projects/EDA_3_2_Gares/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Summarize dataset:   0%| | 0/11 [00:00<?, ?it/s, Describe variable: nb_trains_re
Summarize dataset:   9%| | 1/11 [00:00<00:05,  1.88it/s, Describe variable: nb_t
Summarize dataset:  27%|▎| 3/11 [00:00<00:01,  5.01it/s, Describe variable: nb_t
Export report to file: 100%|█████████████████████| 1/1 [00:00<00:00, 242.11it/s]

✅ Rapport sauvegardé → data/rapport_eda_gares.html
👉 Ouvre ce fichier dans ton navigateur pour explorer.


---
# ÉTAPE 4 — KPIs Métier
> Les 3 KPIs du vocabulaire Infrabel.

In [12]:
derniere_annee = df_gare['date'].dt.year.max()
df_annee = df_gare[df_gare['date'].dt.year == derniere_annee]

# KPI 1 — Ponctualité : % trains avec < 6 min de retard
kpi_ponctualite = df_annee['ponctualite_pct'].mean().round(2)

# KPI 2 — Fiabilité : % trains non annulés
kpi_fiabilite = df_suppression['fiabilite_pct'].mean().round(2)

# KPI 3 — Minutes perdues (impact économique)
kpi_minutes = df_annee['nb_trains_retard'].sum()

print(f'📅 Année analysée     : {derniere_annee}')
print(f'⏱️  Ponctualité globale : {kpi_ponctualite}%  (objectif : 90%)')
print(f'✅  Fiabilité           : {kpi_fiabilite}%')
print(f'🚆  Trains en retard   : {kpi_minutes:,}')

📅 Année analysée     : 2026
⏱️  Ponctualité globale : 94.01%  (objectif : 90%)
✅  Fiabilité           : 96.57%
🚆  Trains en retard   : 68,714.0


---
# ÉTAPE 5 — Visualisation Dashboard
> Les 3 graphiques finaux.

In [13]:
# Graphique 1 — Ponctualité annuelle
df_annuel = df_gare.groupby(df_gare['date'].dt.year)['ponctualite_pct'].mean().reset_index()
df_annuel.columns = ['annee', 'ponctualite_pct']
df_annuel['ponctualite_pct'] = df_annuel['ponctualite_pct'].round(2)

fig = px.bar(
    df_annuel,
    x='annee', y='ponctualite_pct',
    title='📊 Ponctualité Annuelle Moyenne',
    labels={'annee': 'Année', 'ponctualite_pct': 'Ponctualité (%)'},
    text='ponctualite_pct',
    color='ponctualite_pct',
    color_continuous_scale='RdYlGn'
)
fig.add_hline(y=90, line_dash='dash', line_color='red', annotation_text='Objectif 90%')
fig.show()

In [14]:
# Graphique 2 — Responsables des retards
df_top_causes = df_causes.groupby('responsable')['nb_retards'].sum().sort_values(ascending=False).reset_index()

fig = px.bar(
    df_top_causes,
    x='nb_retards', y='responsable',
    orientation='h',
    title='🔍 Responsables des Retards (volume total)',
    labels={'nb_retards': 'Nombre de retards', 'responsable': 'Responsable'},
    color='nb_retards',
    color_continuous_scale='Reds'
)
fig.update_layout(yaxis={'categoryorder': 'total ascending'})
fig.show()

In [15]:
# Graphique 3 — Fiabilité (trains supprimés)
fig = px.line(
    df_suppression,
    x='date', y='fiabilite_pct',
    title='🚫 Fiabilité — % Trains Non Annulés',
    labels={'date': 'Mois', 'fiabilite_pct': 'Fiabilité (%)'},
)
fig.add_hline(y=99, line_dash='dash', line_color='red', annotation_text='Objectif 99%')
fig.show()